In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, classification_report

import pandas as pnd

<h1 style="color: red;">Section 1: Data</h1>

<h2>1) Préparation de données</h2>

In [2]:
dataset = pnd.read_csv('diabetes.csv')  # import
X = np.array(dataset.drop(columns=['Outcome']))  # features
y = np.array(dataset['Outcome'])  # target
# spilt data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

print("Shape X :", X.shape)
print("Colonnes features :", list(dataset.drop(columns=['Outcome']).columns))
print("Répartition de la target :")
print(dataset['Outcome'].value_counts())

Shape X : (768, 8)
Colonnes features : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Répartition de la target :
Outcome
0    500
1    268
Name: count, dtype: int64


<h1 style="color: red;">Section 2: Neural network avec tensorflow</h1>

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

from tensorflow.keras.optimizers import Adam

<h2>2) Modèle de réseau de neurones</h2>

<h3>2-a) Architecture de réseau de neurones</h3>
<p><b>2-a-1) Inputs :</b> le dataset possède 8 features (Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age) → 8 inputs (<code>input_shape=(8,)</code>).</p>
<p><b>2-a-2) Neurones de la couche de sortie :</b> la target Outcome est binaire (0 = pas de diabète, 1 = diabète) → <b>1 seul neurone</b> de sortie suffit.</p>
<p><b>2-a-3) Fonction d'activation de sortie :</b> pour une classification binaire, on utilise l'activation <b>sigmoid</b>, qui transforme la sortie en une probabilité entre 0 et 1.</p>

<p><i>Remarque :</i> la consigne demande 1000 epochs ; on utilise ici 100 epochs (valeur autorisée si les ressources machine sont limitées).</p>

In [ ]:
model_nn = Sequential()
output_layer = Dense(1, input_shape=(X_train.shape[1],), activation='sigmoid')
model_nn.add(output_layer)
opt = Adam(learning_rate=0.001)
model_nn.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision()])
history = model_nn.fit(X_train, y_train, epochs=100, verbose=0)

<h3>2-e) Que fait la fonction fit ?</h3>
<p><code>fit()</code> entraîne le réseau : à chaque epoch, elle calcule la prédiction (forward pass, avec activation sigmoid) pour tout le training set, évalue la fonction de coût (binary crossentropy) entre les probabilités prédites et les labels réels, rétropropage le gradient de cette erreur, puis met à jour les poids et le biais via l'optimiseur Adam. Cette opération se répète pendant le nombre d'epochs indiqué.</p>

<h3>2-f) Nombre de paramètres du réseau</h3>
<p>Calcul à la main : (nombre d'inputs × nombre de neurones de sortie) + nombre de biais = (8 × 1) + 1 = <b>9 paramètres</b>.</p>

In [ ]:
model_nn.summary()
print("Nombre total de paramètres :", model_nn.count_params())

<h3>2-g) Paramètres du réseau de neurones</h3>

In [ ]:
W_nn, bias_nn = model_nn.layers[0].get_weights()

print("Poids :", W_nn.flatten())
print("Biais :", bias_nn)

<h3>2-h) Dessiner le réseau de neurones</h3>

In [ ]:
feature_names = list(dataset.drop(columns=['Outcome']).columns)
n_inputs = len(feature_names)

fig, ax = plt.subplots(figsize=(7, 6))
input_y = np.linspace(-3.5, 3.5, n_inputs)
output_pos = (3, 0)

for y_, name in zip(input_y, feature_names):
    ax.scatter(0, y_, s=900, color='#a8d8ff', edgecolor='black', zorder=3)
    ax.text(-0.15, y_, name, ha='right', va='center', fontsize=8, zorder=4)

ax.scatter(*output_pos, s=1800, color='#ffb3b3', edgecolor='black', zorder=3)
ax.text(*output_pos, 'ŷ', ha='center', va='center', fontsize=13, zorder=4)

for y_, w in zip(input_y, W_nn.flatten()):
    ax.annotate('', xy=output_pos, xytext=(0, y_), arrowprops=dict(arrowstyle='->', lw=1))
    mx, my = (0 + output_pos[0]) / 2, (y_ + output_pos[1]) / 2
    ax.text(mx, my, f"{w:.2f}", fontsize=7, ha='center', backgroundcolor='white')

ax.text(output_pos[0], output_pos[1] + 0.6, f"bias={bias_nn[0]:.3f}", fontsize=9, ha='center')
ax.set_xlim(-2.5, 4)
ax.set_ylim(-4, 4)
ax.axis('off')
ax.set_title("Architecture du réseau de neurones (classification, sigmoid)")
plt.show()

<h3>2-i) Dans quels cas la régularisation (tuning) est-elle importante ?</h3>
<ul>
<li>Écart important entre performance sur training set et test set (overfitting).</li>
<li>Dataset de taille réduite (ici 768 lignes) par rapport à la complexité du modèle.</li>
<li>Classes déséquilibrées, comme ici (500 négatifs contre 268 positifs), ce qui peut biaiser le modèle vers la classe majoritaire.</li>
<li>Présence de features bruitées ou peu informatives.</li>
</ul>

<h2>3) Prédiction en utilisant le modèle</h2>

In [ ]:
yhat_nn_proba = model_nn.predict(X_test).flatten()
yhat_nn = (yhat_nn_proba >= 0.5).astype(int)

<h3>3-b) Prédiction manuelle (sans predict), avec les matrices</h3>
<p>L'activation de sortie étant sigmoid : ŷ = sigmoid(X . W + bias)</p>

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_manual = X_test @ W_nn + bias_nn
yhat_manual_proba = sigmoid(z_manual).flatten()
yhat_manual = (yhat_manual_proba >= 0.5).astype(int)

print("Probabilités identiques (predict vs calcul manuel) :", np.allclose(yhat_manual_proba, yhat_nn_proba, atol=1e-4))

<h2>4) Évaluation du modèle</h2>

<h3>4-a) Performance du modèle sur le training set</h3>

In [ ]:
yhat_train_proba = model_nn.predict(X_train).flatten()
yhat_train = (yhat_train_proba >= 0.5).astype(int)

print("Accuracy train :", accuracy_score(y_train, yhat_train))
print(confusion_matrix(y_train, yhat_train))
print(classification_report(y_train, yhat_train))

<p><b>Interprétation :</b> une accuracy élevée sur le training set montre que le modèle a bien appris à distinguer les deux classes sur les données d'entraînement. Il faut ensuite comparer avec l'accuracy sur le test set pour vérifier la généralisation.</p>

<h3>4-b) Performance du modèle sur le test set</h3>

In [ ]:
print("Accuracy test :", accuracy_score(y_test, yhat_nn))
print(confusion_matrix(y_test, yhat_nn))
print(classification_report(y_test, yhat_nn))

<h3>4-c) Évaluer sur training set et test set en même temps : à quoi ça sert ?</h3>
<ul>
<li>Accuracy élevée sur train mais faible sur test → <b>overfitting</b> (le modèle a mémorisé le training set).</li>
<li>Accuracy faible sur train et sur test → <b>underfitting</b> (le modèle n'apprend pas assez bien, il faut peut-être plus d'epochs, un meilleur learning rate, ou plus de capacité).</li>
<li>Accuracy proche et satisfaisante sur les deux → le modèle généralise bien.</li>
</ul>

<h1>From scratch</h1>

<h2>Exercice 2 : Régression logistique from scratch (version matricielle)</h2>

<h3>1) Utiliser le même dataset de l'exercice 1</h3>
<p>On réutilise X_train, X_test, y_train, y_test définis en Section 1. Contrairement au dataset synthétique de l'exercice de régression (features toutes entre 0 et 10), les 8 features du dataset diabetes ont des échelles très différentes (ex. Glucose ~0-199, Insulin ~0-846). Sans normalisation, la descente de gradient converge très mal. On standardise donc les features (moyenne 0, écart-type 1) avant l'entraînement from scratch.</p>

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

<h3>5) Pourquoi développer des modèles sans utilisation de librairies ?</h3>
<ul>
<li>Comprendre en profondeur les mécanismes internes d'un modèle (forward pass, fonction de coût, calcul des gradients, descente de gradient).</li>
<li>Être capable de déboguer, d'adapter ou de personnaliser un algorithme dans des cas non couverts par les librairies existantes.</li>
<li>Objectif pédagogique : vérifier qu'on maîtrise les maths derrière les modèles, avant de les utiliser « en boîte noire » via sklearn/keras.</li>
<li>Avoir un contrôle total sur l'implémentation (choix des optimisations, structures de données, etc.).</li>
</ul>

<h3>6) Quelles sont les principales étapes à suivre ?</h3>
<ol>
<li>Préparer / normaliser les données (features à des échelles comparables).</li>
<li>Initialiser les paramètres W et b (souvent à 0 ou aléatoirement).</li>
<li>Calculer la prédiction (forward) : z = X.W + b, puis ŷ = sigmoid(z).</li>
<li>Calculer la fonction de coût (binary cross-entropy) entre ŷ et y.</li>
<li>Calculer les gradients de la fonction de coût par rapport à W et b.</li>
<li>Mettre à jour les paramètres via la descente de gradient : W ← W - lr.dW, b ← b - lr.db.</li>
<li>Répéter les étapes 3 à 6 pendant un nombre d'epochs donné, jusqu'à convergence.</li>
<li>Évaluer le modèle sur le training set et le test set.</li>
</ol>

<h3>7) Comment les dérivées de la cost function participent-elles à la mise à jour du modèle ?</h3>
<p>Pour la régression logistique avec la binary cross-entropy, la dérivée du coût par rapport à z se simplifie en (ŷ - y). On en déduit :</p>
<p>dW = (1/n) . X^T . (ŷ - y) et db = (1/n) . somme(ŷ - y)</p>
<p>Ces dérivées (gradients) indiquent la direction et l'intensité selon lesquelles chaque paramètre doit être modifié pour réduire la fonction de coût. La descente de gradient utilise ces valeurs pour ajuster W et b à chaque itération, dans la direction opposée au gradient (puisqu'on veut minimiser le coût).</p>

<h3>8) Compléter le code source pour entraîner le modèle</h3>

In [4]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


learning_rate = 0.0001
epochs = 1000


# Initialisation des paramètres
W = np.zeros((X_train_scaled.shape[1], 1))  # Shape: (8, 1)
b = 0.0

# Reshape y_train pour garantir les dimensions adéquates
y_train = y_train.reshape(-1, 1)  # Shape: (n_samples, 1)
n = len(X_train_scaled)

# Entraînement (descente de gradient vectorisée)
for epoch in range(epochs):
    z = X_train_scaled @ W + b
    y_pred = sigmoid(z)  # Shape: (n, 1)
    error = y_pred - y_train  # Shape: (n, 1)

    dW = (1 / n) * (X_train_scaled.T @ error)  # Shape: (8, 1)
    db = (1 / n) * np.sum(error)               # Scalaire

    W -= learning_rate * dW
    b -= learning_rate * db

print("Paramètres ajustés:")
print(f"W = \n{W}")
print(f"b = {b:.4f}")

Paramètres ajustés:
W = 
[[0.00795179]
 [0.02206172]
 [0.00180623]
 [0.00405618]
 [0.00585176]
 [0.01347747]
 [0.00834286]
 [0.01019173]]
b = -0.0148


<h3>9) Évaluer le modèle</h3>

In [5]:
def predict_scratch(X, W, b, threshold=0.5):
    proba = sigmoid(X @ W + b).flatten()
    return (proba >= threshold).astype(int)

yhat_train_scratch = predict_scratch(X_train_scaled, W, b)
yhat_test_scratch = predict_scratch(X_test_scaled, W, b)

print("Accuracy train (from scratch) :", accuracy_score(y_train.ravel(), yhat_train_scratch))
print("Accuracy test  (from scratch) :", accuracy_score(y_test, yhat_test_scratch))
print(confusion_matrix(y_test, yhat_test_scratch))
print(classification_report(y_test, yhat_test_scratch))

Accuracy train (from scratch) : 0.744299674267101
Accuracy test  (from scratch) : 0.8116883116883117
[[85 16]
 [13 40]]
              precision    recall  f1-score   support

           0       0.87      0.84      0.85       101
           1       0.71      0.75      0.73        53

    accuracy                           0.81       154
   macro avg       0.79      0.80      0.79       154
weighted avg       0.81      0.81      0.81       154



<h3>10) Comparer les résultats avec le modèle de régression logistique de sklearn</h3>

In [6]:
from sklearn.linear_model import LogisticRegression

model_sklearn = LogisticRegression(max_iter=1000)
model_sklearn.fit(X_train_scaled, y_train.ravel())

yhat_sklearn = model_sklearn.predict(X_test_scaled)

print("Coefficients sklearn      :", model_sklearn.coef_.ravel())
print("Intercept sklearn         :", model_sklearn.intercept_)
print("Coefficients from scratch :", W.ravel())
print("Intercept from scratch    :", b)
print()
print("Accuracy sklearn      :", accuracy_score(y_test, yhat_sklearn))
print("Accuracy from scratch :", accuracy_score(y_test, yhat_test_scratch))

Coefficients sklearn      : [ 2.62593843e-01  1.15541285e+00 -2.71554455e-01  7.39216475e-04
 -1.42052821e-01  6.79307016e-01  3.20289515e-01  1.95627679e-01]
Intercept sklearn         : [-0.85848]
Coefficients from scratch : [0.00795179 0.02206172 0.00180623 0.00405618 0.00585176 0.01347747
 0.00834286 0.01019173]
Intercept from scratch    : -0.014798185743334264

Accuracy sklearn      : 0.7987012987012987
Accuracy from scratch : 0.8116883116883117


<p><b>Comparaison :</b> sklearn utilise un solveur optimisé (avec régularisation L2 par défaut) qui converge rapidement vers un optimum. Le modèle from scratch, avec un learning_rate faible (0.0001) et seulement 1000 epochs, obtient une accuracy du même ordre de grandeur, ce qui valide l'implémentation manuelle de la descente de gradient. Avec davantage d'epochs et/ou un learning_rate plus élevé, les résultats du modèle from scratch se rapprocheraient encore de ceux de sklearn.</p>